L2-regularized logistic regression for binary or multiclass classification; trains a model (on `train.txt`), optimizes L2 regularization strength on `dev.txt`, and evaluates performance on `test.txt`.  Reports test accuracy with 95% confidence intervals and prints out the strongest coefficients for each class.

In [ ]:
from scipy import sparse
from sklearn import linear_model
from collections import Counter
import numpy as np
import operator
import nltk
import math
import string
from scipy.stats import norm

In [ ]:
!python -m nltk.downloader punkt

<frozen runpy>:128: RuntimeWarning: 'nltk.downloader' found in sys.modules after import of package 'nltk', but prior to execution of 'nltk.downloader'; this may result in unpredictable behaviour
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [ ]:
def load_data(filename):
    X = []
    Y = []
    with open(filename, encoding="utf-8") as file:
        for line in file:
            cols = line.strip().split("\t")
            if len(cols) < 3:
                continue
            if cols[0] == "id" and cols[1] == "label":
                continue

            idd = cols[0]
            label = cols[1].strip()
            text = cols[2]

            X.append(text)
            Y.append(label)

    return X, Y


In [ ]:
class Classifier:

    def __init__(self, feature_method, trainX, trainY, devX, devY, testX, testY):
        self.feature_vocab = {}
        self.feature_method = feature_method
        self.min_feature_count=10 #changing this to 10 increased accuracy by 3%
        self.log_reg = None

        self.trainY=trainY
        self.devY=devY
        self.testY=testY

        self.trainX = self.process(trainX, training=True)
        self.devX = self.process(devX, training=False)
        self.testX = self.process(testX, training=False)

    # Featurize entire dataset
    def featurize(self, data):
        featurized_data = []
        for text in data:
            feats = self.feature_method(text)
            featurized_data.append(feats)
        return featurized_data

    # Read dataset and returned featurized representation as sparse matrix + label array
    def process(self, X_data, training = False):

        data = self.featurize(X_data)

        if training:
            fid = 0
            feature_doc_count = Counter()
            for feats in data:
                for feat in feats:
                    feature_doc_count[feat]+= 1

            for feat in feature_doc_count:
                if feature_doc_count[feat] >= self.min_feature_count:
                    self.feature_vocab[feat] = fid
                    fid += 1

        F = len(self.feature_vocab)
        D = len(data)
        X = sparse.dok_matrix((D, F))
        for idx, feats in enumerate(data):
            for feat in feats:
                if feat in self.feature_vocab:
                    X[idx, self.feature_vocab[feat]] = feats[feat]

        return X

    # Train model and evaluate on held-out data
    def train(self):
        (D,F) = self.trainX.shape
        best_dev_accuracy=0
        best_model=None
        for C in [0.1, 1, 10, 100]:
            self.log_reg = linear_model.LogisticRegression(C = C, max_iter=1000)
            self.log_reg.fit(self.trainX, self.trainY)
            training_accuracy = self.log_reg.score(self.trainX, self.trainY)
            development_accuracy = self.log_reg.score(self.devX, self.devY)
            if development_accuracy > best_dev_accuracy:
                best_dev_accuracy=development_accuracy
                best_model=self.log_reg

#             print("C: %s, Train accuracy: %.3f, Dev accuracy: %.3f" % (C, training_accuracy, development_accuracy))

        self.log_reg=best_model


    def test(self):
        return self.log_reg.score(self.testX, self.testY)


    def printWeights(self, n=10):

        reverse_vocab=[None]*len(self.log_reg.coef_[0])
        for k in self.feature_vocab:
            reverse_vocab[self.feature_vocab[k]]=k

        # binary
        if len(self.log_reg.classes_) == 2:
              weights=self.log_reg.coef_[0]

              cat=self.log_reg.classes_[1]
              for feature, weight in list(reversed(sorted(zip(reverse_vocab, weights), key = operator.itemgetter(1))))[:n]:
                  print("%s\t%.3f\t%s" % (cat, weight, feature))
              print()

              cat=self.log_reg.classes_[0]
              for feature, weight in list(sorted(zip(reverse_vocab, weights), key = operator.itemgetter(1)))[:n]:
                  print("%s\t%.3f\t%s" % (cat, weight, feature))
              print()

        # multiclass
        else:
          for i, cat in enumerate(self.log_reg.classes_):

              weights=self.log_reg.coef_[i]

              for feature, weight in list(reversed(sorted(zip(reverse_vocab, weights), key = operator.itemgetter(1))))[:n]:
                  print("%s\t%.3f\t%s" % (cat, weight, feature))
              print()



In [ ]:
import nltk
from nltk.util import ngrams
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from collections import Counter
import string
nltk.download('vader_lexicon')
VADER = SentimentIntensityAnalyzer()

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


In [ ]:
import re
from nltk import bigrams
#code referenced from https://www.nltk.org/api/nltk.sentiment.vader.html

def binary_bow_featurize(text):
    feats = {}
    words = nltk.word_tokenize(text)

    #remove ',', ';' , ''' and '.' within the tokens
    #improved test accuracy by 1%
    clean_words = [word.strip("',;.") for word in words]

    for word in clean_words:
      #word=word.lower() #got better accuracy (+4%) commenting this out
      feats[word]= 1

    #returns a dict with 'pos', 'neg', 'neu', and 'compound' scores using VADER
    #improved test accuracy by 4%
    sentiment = VADER.polarity_scores(text)
    feats["sentiment_pos"] = sentiment['pos']
    feats["sentiment_neg"] = sentiment['neg']
    #feats["sentiment_neu"] = sentiment['neu'] #improve test accuracy by 1% when we don't use neu
    feats["sentiment_compound"] = sentiment['compound']

    #we also tried other adding other features but they decreaesed the test accuracy
    #ex: bigrams decreased accuracy by 5%
    # for word in bigrams(clean_words):
    #     feats[f"{word[0]} {word[1]}"] = 1

    return feats

In [ ]:
def confidence_intervals(accuracy, n, significance_level):
    critical_value=(1-significance_level)/2
    z_alpha=-1*norm.ppf(critical_value)
    se=math.sqrt((accuracy*(1-accuracy))/n)
    return accuracy-(se*z_alpha), accuracy+(se*z_alpha)

In [ ]:
def run(trainingFile, devFile, testFile):
    trainX, trainY=load_data(trainingFile)
    devX, devY=load_data(devFile)
    testX, testY=load_data(testFile)

    simple_classifier = Classifier(binary_bow_featurize, trainX, trainY, devX, devY, testX, testY)
    simple_classifier.train()
    accuracy=simple_classifier.test()

    lower, upper=confidence_intervals(accuracy, len(testY), .95)
    print("Test accuracy for best dev model: %.3f, 95%% CIs: [%.3f %.3f]\n" % (accuracy, lower, upper))

    simple_classifier.printWeights()


In [ ]:
trainingFile = '/content/sample_data/train.txt'
devFile = '/content/sample_data/dev.txt'
testFile = '/content/sample_data/test.txt'
run(trainingFile, devFile, testFile)
#improved logreg accuracy for best dev model: 0.440, 95% CIs: [0.343 0.537]

Test accuracy for best dev model: 0.440, 95% CIs: [0.343 0.537]

1	0.145	opportunity
1	0.141	times
1	0.141	short
1	0.135	learning
1	0.134	same
1	0.132	think
1	0.126	without
1	0.126	out
1	0.122	other
1	0.119	from

2	0.378	I
2	0.235	good
2	0.216	will
2	0.210	better
2	0.199	every
2	0.166	point
2	0.162	has
2	0.153	which
2	0.149	pay
2	0.144	who

3	0.250	its
3	0.206	Moreover
3	0.189	students
3	0.182	those
3	0.180	amount
3	0.177	between
3	0.175	course
3	0.174	true
3	0.168	no
3	0.161	much

4	0.307	provide
4	0.278	because
4	0.275	First
4	0.257	by
4	0.256	On
4	0.224	job
4	0.221	friends
4	0.208	living
4	0.203	another
4	0.199	up

5	0.509	example
5	0.330	For
5	0.317	such
5	0.293	are
5	0.245	work
5	0.231	used
5	0.225	when
5	0.213	what
5	0.212	I
5	0.205	t

6	0.198	countries
6	0.167	time
6	0.154	waste
6	0.153	at
6	0.145	sports
6	0.137	free
6	0.131	provides
6	0.128	important
6	0.127	home
6	0.126	In

7	0.190	we
7	0.173	In
7	0.169	themselves
7	0.165	skills
7	0.149	which
7	0.149	might
7	0.149	most
7	0.145

In [249]:
def binary_bow_featurize_original(text):
    feats = {}
    words = nltk.word_tokenize(text)

    for word in words:
        word=word.lower()
        feats[word]=1

    return feats

In [250]:
def run(trainingFile, devFile, testFile):
    trainX, trainY=load_data(trainingFile)
    devX, devY=load_data(devFile)
    testX, testY=load_data(testFile)

    simple_classifier = Classifier(binary_bow_featurize_original, trainX, trainY, devX, devY, testX, testY)
    simple_classifier.train()
    accuracy=simple_classifier.test()

    lower, upper=confidence_intervals(accuracy, len(testY), .95)
    print("Test accuracy for best dev model: %.3f, 95%% CIs: [%.3f %.3f]\n" % (accuracy, lower, upper))

    simple_classifier.printWeights()


In [251]:
trainingFile = '/content/sample_data/train.txt'
devFile = '/content/sample_data/dev.txt'
testFile = '/content/sample_data/test.txt'
run(trainingFile, devFile, testFile)

Test accuracy for best dev model: 0.330, 95% CIs: [0.238 0.422]

1	0.686	'
1	0.676	think
1	0.633	opportunity
1	0.626	person
1	0.550	is
1	0.540	short
1	0.538	times
1	0.501	kind
1	0.481	learning
1	0.476	same

2	1.353	i
2	0.932	good
2	0.891	better
2	0.888	every
2	0.874	has
2	0.797	our
2	0.795	will
2	0.765	finally
2	0.748	point
2	0.723	done

3	1.075	its
3	0.923	amount
3	0.907	between
3	0.897	true
3	0.888	much
3	0.839	;
3	0.813	course
3	0.791	rather
3	0.773	however
3	0.749	cars

4	1.601	job
4	1.476	provide
4	1.299	research
4	1.227	by
4	1.190	another
4	1.165	instead
4	1.163	living
4	1.158	had
4	1.149	first
4	1.100	without

5	2.303	example
5	1.570	used
5	1.187	support
5	1.159	work
5	1.141	must
5	1.102	play
5	1.079	there
5	1.047	world
5	1.039	such
5	1.020	any

6	0.866	countries
6	0.725	someone
6	0.725	waste
6	0.633	free
6	0.627	sports
6	0.605	these
6	0.605	well
6	0.568	time
6	0.542	been
6	0.541	most

7	0.847	about
7	0.819	we
7	0.773	instance
7	0.766	those
7	0.761	animals
7	0.732	because
7	0.70